# Credit Card Default Prediction — EDA & Model Building

Dataset: **Default of Credit Card Clients** (30,000 observations, 24 predictors + target).

Goal: explore the data, understand default behavior, preprocess it, train classification models, evaluate them, and identify important predictors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay

CSV_PATH = r'converted/default of credit card clients.csv'
raw = pd.read_csv(CSV_PATH, header=None)
columns = raw.iloc[1].tolist()
df = raw.iloc[2:].copy()
df.columns = columns
df = df.reset_index(drop=True)
df.head()

## 1. Basic data inspection

In [ ]:
print('Shape:', df.shape)
display(df.info())
display(df.describe().T)
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())


## 2. Target distribution

The target is `default payment next month`, where 1 means default and 0 means no default.

In [ ]:
target = 'default payment next month'
print(df[target].value_counts())
print(df[target].value_counts(normalize=True).mul(100).round(2))
sns.countplot(data=df, x=target)
plt.title('Next-Month Default Distribution')
plt.show()

## 3. Univariate / categorical EDA

Important coded categorical variables include sex, education, marriage, and payment-status fields.

In [ ]:
categorical = ['SEX', 'EDUCATION', 'MARRIAGE']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, categorical):
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Numeric distributions

In [ ]:
numeric = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3']
df[numeric].hist(figsize=(14, 10), bins=30)
plt.suptitle('Selected Numeric Feature Distributions')
plt.tight_layout()
plt.show()

## 5. Default rate by important categorical variables

In [ ]:
for col in ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3']:
    rates = df.groupby(col)[target].mean().sort_values(ascending=False)
    display(rates.rename('default_rate').to_frame())
    rates.plot(kind='bar', figsize=(7, 3), title=f'Default Rate by {col}')
    plt.ylabel('Default rate')
    plt.show()

## 6. Correlation analysis

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

print('Strongest negative correlations with target:')
display(corr[target].drop(target).sort_values().head(10).to_frame('correlation'))
print('Strongest positive correlations with target:')
display(corr[target].drop(target).sort_values(ascending=False).head(10).to_frame('correlation'))

## 7. Preprocessing and train/test split

`ID` is excluded because it is an identifier rather than a predictive feature. The target is moderately imbalanced, so stratified splitting is used.

In [ ]:
X = df.drop(columns=[target, 'ID'])
y = df[target].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print('Train default rate:', y_train.mean())
print('Test default rate:', y_test.mean())

## 8. Model 1 — Logistic Regression

In [ ]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:, 1]
print(classification_report(y_test, lr_pred))
print('ROC-AUC:', roc_auc_score(y_test, lr_prob))

## 9. Model 2 — Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=250, random_state=42, n_jobs=-1, class_weight='balanced'
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]
print(classification_report(y_test, rf_pred))
print('ROC-AUC:', roc_auc_score(y_test, rf_prob))

## 10. Model 3 — HistGradientBoosting

In [ ]:
hgb = HistGradientBoostingClassifier(
    max_iter=250, learning_rate=0.08, max_leaf_nodes=31, random_state=42
)
hgb.fit(X_train, y_train)
hgb_pred = hgb.predict(X_test)
hgb_prob = hgb.predict_proba(X_test)[:, 1]
print(classification_report(y_test, hgb_pred))
print('ROC-AUC:', roc_auc_score(y_test, hgb_prob))

## 11. Compare models

In [ ]:
results = []
for name, pred, prob in [
    ('Logistic Regression', lr_pred, lr_prob),
    ('Random Forest', rf_pred, rf_prob),
    ('HistGradientBoosting', hgb_pred, hgb_prob)
]:
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
display(results_df.round(4))

## 12. Confusion matrix and ROC curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name, pred in zip(axes, ['Logistic Regression', 'Random Forest', 'HistGradientBoosting'], [lr_pred, rf_pred, hgb_pred]):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
for name, prob in [('Logistic Regression', lr_prob), ('Random Forest', rf_prob), ('HistGradientBoosting', hgb_prob)]:
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=plt.gca())
plt.plot([0,1], [0,1], '--', label='Chance')
plt.title('ROC Curves')
plt.legend()
plt.show()

## 13. Feature importance

For the tree models, feature importance can be used to identify variables that contribute most to prediction.

In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance.head(15).to_frame('importance'))
importance.head(15).sort_values().plot(kind='barh', figsize=(8, 6))
plt.title('Top Random Forest Feature Importances')
plt.xlabel('Importance')
plt.show()

## Key interpretation

- The dataset contains **30,000 customers** and **24 usable predictors** after removing `ID`.
- The default class is the minority class (about **22.12%**), so accuracy alone is not sufficient.
- Payment-status variables, especially recent repayment status (`PAY_0`, `PAY_2`, etc.), tend to have the strongest positive relationship with default.
- `LIMIT_BAL` has a negative relationship with default, meaning lower credit limits are generally associated with higher observed default risk.
- In the baseline comparison, tree-based models generally provide stronger recall/F1 and ROC-AUC than the basic logistic-regression model.
- For a real credit-risk system, threshold tuning, calibration, fairness checks, and cross-validation should be added before deployment.